In [ ]:
import numpy as np

def is_probability_vector(p, tol=1e-8):
    p = np.asarray(p, dtype=float)
    return np.all(p >= -tol) and np.isclose(p.sum(), 1.0, atol=tol)

def empirical_distribution(samples, n_outcomes):
    samples = np.asarray(samples, dtype=int)
    counts = np.bincount(samples, minlength=n_outcomes).astype(float)
    return counts / counts.sum()

def expected_value(values, probs):
    values = np.asarray(values, dtype=float)
    probs = np.asarray(probs, dtype=float)
    if not is_probability_vector(probs):
        raise ValueError("probs must be nonnegative and sum to one")
    return float(np.dot(values, probs))

def bernoulli_standard_error(p_hat, shots):
    return np.sqrt(p_hat * (1.0 - p_hat) / shots)

def pauli_expectation_from_counts(n_plus, n_minus):
    shots = n_plus + n_minus
    mu_hat = (n_plus - n_minus) / shots
    se_hat = np.sqrt(max(0.0, 1.0 - mu_hat**2) / shots)
    return mu_hat, se_hat

def total_variation(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    return 0.5 * np.abs(p - q).sum()


In [ ]:
# Runtime checks for the Appendix II listing.
p = np.array([0.1, 0.2, 0.3, 0.4])
q = np.array([0.25, 0.25, 0.25, 0.25])
samples = np.array([0, 1, 1, 2, 2, 2])
dist = empirical_distribution(samples, n_outcomes=4)

assert is_probability_vector(p)
assert is_probability_vector(q)
assert is_probability_vector(dist)
assert not is_probability_vector([0.2, 0.2])
assert not is_probability_vector([0.5, -0.1, 0.6])
assert np.allclose(dist, [1 / 6, 2 / 6, 3 / 6, 0.0])
assert np.isclose(expected_value([0, 1, 2, 3], p), 2.0)
assert np.isclose(bernoulli_standard_error(0.25, 100), np.sqrt(0.25 * 0.75 / 100))

mu_hat, se_hat = pauli_expectation_from_counts(n_plus=620, n_minus=380)
assert np.isclose(mu_hat, 0.24)
assert np.isclose(se_hat, np.sqrt((1.0 - 0.24**2) / 1000))
assert np.isclose(total_variation(p, q), 0.2)

try:
    expected_value([0, 1], [0.2, 0.2])
except ValueError:
    pass
else:
    raise AssertionError("expected_value should reject invalid probability vectors")

print("Appendix II checks passed.")
